# Exploratory Data Analysis and Interactive Visualisation

**Data Science & Machine Learning: from Skills to the Project** · 90 minutes

**Case study: customer churn.** A subscription business loses customers who cancel. We want to understand which customers churn and what distinguishes them, using systematic exploratory data analysis (EDA) and interactive charts.

The dataset is **synthetic**. It is generated inside this notebook with `numpy.random.default_rng(42)` and does not describe any real customers. Every pattern you find describes this sample only. Run the cells from top to bottom.

## Learning outcomes

By the end of the session you should be able to:

- conduct a systematic EDA, from data audit to group-level comparison;
- analyse data quality, distributions, outliers, correlations and group differences;
- visualise multivariate relationships without overwhelming the reader;
- create interactive Plotly charts and use them for investigation;
- communicate findings that are defensible, with their uncertainty and limitations stated.

## Agenda (90 minutes)

| Time | Section | Focus |
|---|---|---|
| 5 min | 1. Business problem and workflow | Questions about churn; Question → Visualisation → Interpretation → Validation → Decision |
| 10 min | 2. Initial data audit | Shape, types, missing values, duplicates, category labels |
| 15 min | 3. Distribution analysis | Shape, outliers, group differences, limits of summary statistics |
| 15 min | 4. Relationships and correlations | Scatter plots, Pearson versus Spearman, misleading aggregates |
| 10 min | 5. Multivariate exploration | Hue, size, faceting, pairplots, why 3D charts mislead |
| 10 min | 6. Interactive Plotly | Hover, zoom and legend filtering for investigation |
| 5 min | 7. Responsible visual communication | Colour, labels, uncertainty, limits of synthetic data |
| 15 min | 8. Guided mini-project | Three TODO questions on churn |
| 5 min | 9. Findings and reflection | Evidence-based findings template and reflection |

## Setup: the synthetic churn dataset

The next cell generates about 500 customer records, reproducibly, with `numpy.random.default_rng(42)`. The structure is deliberately plausible:

- churn is more likely for month-to-month contracts, shorter tenure and more support tickets;
- premium customers spend more, commit more often and churn less;
- tenure depends on contract type.

The cell also adds **controlled data-quality issues**: missing values, a few extreme values, inconsistent labels and duplicated rows. Real datasets contain problems like these, and the audit in the next section finds them. The issues are teaching devices, not facts about any real business.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid', context='notebook')

# Okabe-Ito palette: stays distinguishable for most forms of colour blindness
OKABE_ITO = ['#0072B2', '#E69F00', '#009E73', '#D55E00', '#CC79A7', '#56B4E9', '#F0E442', '#000000']
CHURN_COLOURS = {'Retained': OKABE_ITO[0], 'Churned': OKABE_ITO[3]}
PLAN_COLOURS = {'Basic': OKABE_ITO[5], 'Standard': OKABE_ITO[1], 'Premium': OKABE_ITO[4]}
PLAN_ORDER = ['Basic', 'Standard', 'Premium']
CONTRACT_ORDER = ['Month-to-month', 'One year', 'Two year']

rng = np.random.default_rng(42)
n = 500
as_of = pd.Timestamp('2026-06-30')          # reference date for the synthetic extract

plan_type = rng.choice(PLAN_ORDER, size=n, p=[0.50, 0.32, 0.18])

# Contract choice depends on plan: premium customers commit more often
contract_probs = {'Basic': [0.60, 0.25, 0.15], 'Standard': [0.40, 0.35, 0.25], 'Premium': [0.25, 0.35, 0.40]}
contract_type = np.empty(n, dtype=object)
for plan in PLAN_ORDER:
    idx = np.flatnonzero(plan_type == plan)
    contract_type[idx] = rng.choice(CONTRACT_ORDER, size=idx.size, p=contract_probs[plan])

# Tenure depends on contract: longer commitments, longer relationships (right-skewed gamma draws)
shape = pd.Series(contract_type).map({'Month-to-month': 2, 'One year': 3, 'Two year': 4}).to_numpy()
scale = pd.Series(contract_type).map({'Month-to-month': 5, 'One year': 8, 'Two year': 10}).to_numpy()
tenure_months = np.clip(np.round(rng.gamma(shape, scale)), 1, 72).astype(int)

# Monthly spend depends on plan, with multiplicative variation
base_spend = pd.Series(plan_type).map({'Basic': 22, 'Standard': 42, 'Premium': 78}).to_numpy()
monthly_spend = (base_spend * rng.lognormal(0, 0.2, size=n)).round(2)

# Support tickets depend on plan (basic customers contact support more often)
ticket_rate = pd.Series(plan_type).map({'Basic': 1.8, 'Standard': 1.2, 'Premium': 0.8}).to_numpy()
support_tickets = rng.poisson(ticket_rate)

# Churn: logistic model with contract, tenure, support tickets and plan as drivers
logit = (-1.0
         + 1.0 * (contract_type == 'Month-to-month')
         - 0.04 * tenure_months
         + 0.35 * support_tickets
         - 0.5 * (plan_type == 'Premium'))
churn_probability = 1 / (1 + np.exp(-logit))
churned = (rng.random(n) < churn_probability).astype(int)

signup_date = as_of - pd.to_timedelta(tenure_months * 30 + rng.integers(0, 30, size=n), unit='D')

df = pd.DataFrame({
    'customer_id': [f'C{i:04d}' for i in range(1, n + 1)],
    'plan_type': plan_type,
    'tenure_months': tenure_months,
    'monthly_spend': monthly_spend,
    'support_tickets': support_tickets,
    'contract_type': contract_type,
    'signup_date': signup_date,
    'churned': churned,
})

# Controlled data-quality issues (synthetic)
df.loc[rng.choice(n, size=int(0.06 * n), replace=False), 'monthly_spend'] = np.nan       # missing spend
df.loc[rng.choice(n, size=int(0.04 * n), replace=False), 'support_tickets'] = np.nan     # missing tickets
df.loc[rng.choice(n, size=5, replace=False), 'contract_type'] = np.nan                  # missing contract

spend_rows = rng.choice(np.flatnonzero(df['monthly_spend'].notna().to_numpy()), size=3, replace=False)
df.loc[spend_rows, 'monthly_spend'] *= 10                                               # likely billing errors

ticket_rows = rng.choice(np.flatnonzero(df['support_tickets'].notna().to_numpy()), size=2, replace=False)
df.loc[ticket_rows, 'support_tickets'] = [35, 42]                                       # implausible counts

label_rows = rng.choice(np.flatnonzero((df['contract_type'] == 'Month-to-month').to_numpy()), size=6, replace=False)
df.loc[label_rows, 'contract_type'] = 'month-to-month'                                  # label variant

duplicate_rows = rng.choice(n, size=3, replace=False)
df = pd.concat([df, df.iloc[duplicate_rows]], ignore_index=True)                       # duplicated records

print(f'Generated {len(df)} rows (synthetic) with {df.shape[1]} columns.')

## 2. Initial data audit

Before any plot, audit the data. Check the shape, column types, a sample of rows, the summary statistics, missing values, duplicates and the frequency of each category.

Do **not** remove suspicious records automatically. A missing value, a very large spend or a duplicated row may be an error, a genuine unusual customer, or a symptom of a pipeline problem. The audit documents them; a deliberate decision comes later.

In [ ]:
print('Shape:', df.shape)
print()
df.info()
print()
df.head()

In [ ]:
df.describe(include='all').T

In [ ]:
spend = df['monthly_spend']
q1, q3 = spend.quantile([0.25, 0.75])
fence = q3 + 3 * (q3 - q1)              # far-outlier fence for monthly_spend

checks = {
    'Duplicated customer_id (rows)': df['customer_id'].duplicated().sum(),
    'Exact duplicate rows': df.duplicated().sum(),
    'Missing monthly_spend': spend.isna().sum(),
    'Missing support_tickets': df['support_tickets'].isna().sum(),
    'Missing contract_type': df['contract_type'].isna().sum(),
    'Lower-case contract label variant': (df['contract_type'] == 'month-to-month').sum(),
    f'monthly_spend above far-outlier fence (£{fence:,.0f})': (spend > fence).sum(),
    'support_tickets above 20': (df['support_tickets'] > 20).sum(),
}
quality_summary = pd.DataFrame({'count': checks})
quality_summary['% of rows'] = (100 * quality_summary['count'] / len(df)).round(1)
quality_summary

In [ ]:
print('Plan type frequencies:')
print(df['plan_type'].value_counts(dropna=False).to_string())
print()
print('Contract type frequencies:')
print(df['contract_type'].value_counts(dropna=False).to_string())
print()
print('Churned (0 = retained, 1 = churned):')
print(df['churned'].value_counts(dropna=False).to_string())

**Questions to answer from the audit:**

- Which columns have missing values, and roughly how much? Is the missingness likely to be random, or could it depend on another variable?
- Are the duplicated rows identical, or only the ID repeats? What would happen to your averages if you kept them?
- Are the extreme spend and ticket values plausible? What evidence would you need before deciding?
- Why might `support_tickets` have become a decimal-valued column?

The next cell standardises one label issue. This is a decision: the two spellings name the same contract type, so we merge them. We keep every record and add a readable label for the churn flag.

In [ ]:
df['contract_type'] = df['contract_type'].replace({'month-to-month': 'Month-to-month'})
df['churn_label'] = df['churned'].map({0: 'Retained', 1: 'Churned'})
print(df['contract_type'].value_counts(dropna=False).to_string())

## 3. Distribution analysis

Distributions show shape, spread and outliers. Summary statistics such as the mean and standard deviation assume roughly symmetric, single-peaked data. Skewed or multimodal data can make them misleading, and a group difference in averages can hide a difference in shape.

Use histograms and KDEs to see the shape of one variable, and box or violin plots to compare groups.

First, the shape of `tenure_months` and `monthly_spend` across all customers:

In [ ]:
print(df[['tenure_months', 'monthly_spend', 'support_tickets']].skew().round(2).to_string())
print()

fig, (ax_tenure, ax_spend) = plt.subplots(1, 2, figsize=(11, 4))

sns.histplot(df['tenure_months'], bins=24, kde=True, color=OKABE_ITO[0], ax=ax_tenure)
ax_tenure.set_title('Tenure: histogram with KDE')
ax_tenure.set_xlabel('Tenure (months)')
ax_tenure.set_ylabel('Number of customers')

sns.histplot(df['monthly_spend'].dropna(), bins=40, kde=True, color=OKABE_ITO[2], ax=ax_spend)
ax_spend.set_title('Monthly spend: histogram with KDE')
ax_spend.set_xlabel('Monthly spend (£)')
ax_spend.set_ylabel('Number of customers')
fig.tight_layout()

Next, the same variables compared between retained and churned customers. Box plots show medians, quartiles and extreme values; violin plots show the full shape.

Look at the whole picture, not just the centre: the shape of each group and the location of its outliers can differ even when the medians are close.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))

sns.boxplot(data=df, x='churn_label', y='tenure_months', hue='churn_label', palette=CHURN_COLOURS,
            legend=False, order=['Retained', 'Churned'], ax=axes[0, 0])
axes[0, 0].set_title('Tenure by churn status: box plot')
axes[0, 0].set_ylabel('Tenure (months)')

sns.violinplot(data=df, x='churn_label', y='tenure_months', hue='churn_label', palette=CHURN_COLOURS,
               legend=False, order=['Retained', 'Churned'], inner='quartile', ax=axes[0, 1])
axes[0, 1].set_title('Tenure by churn status: violin plot')
axes[0, 1].set_ylabel('Tenure (months)')

sns.boxplot(data=df, x='churn_label', y='monthly_spend', hue='churn_label', palette=CHURN_COLOURS,
            legend=False, order=['Retained', 'Churned'], ax=axes[1, 0])
axes[1, 0].set_title('Monthly spend by churn status: box plot')
axes[1, 0].set_ylabel('Monthly spend (£)')

sns.violinplot(data=df, x='churn_label', y='monthly_spend', hue='churn_label', palette=CHURN_COLOURS,
               legend=False, order=['Retained', 'Churned'], inner='quartile', ax=axes[1, 1])
axes[1, 1].set_title('Monthly spend by churn status: violin plot')
axes[1, 1].set_ylabel('Monthly spend (£)')

for ax in axes.flat:
    ax.set_xlabel('')
fig.tight_layout()

**Questions to discuss:**

- Does `tenure_months` show one peak or several? If there are several, what mixture of customers could produce them?
- How does the skew of `monthly_spend` change the relationship between its mean and median?
- Are the extreme spend values errors or genuine customers? Which chart best shows where they sit, and what would you check next?
- Do the two churn groups differ more in their centre, their spread or their shape?

## 4. Relationships and correlations

Two common measures of association are:

- **Pearson correlation** measures the strength of a *linear* relationship. It is sensitive to outliers.
- **Spearman correlation** uses ranks. It measures monotonic relationships and is more robust to outliers and skew.

If the two disagree, the relationship may be non-linear or outlier-driven. Examine a scatter plot before trusting either number.

A correlation between two variables is not causation. A third variable, such as plan type, can produce both. Investigate whether the association persists within groups.

First, a scatter plot of tenure against monthly spend, coloured by churn status:

In [ ]:
scatter_data = df.dropna(subset=['tenure_months', 'monthly_spend'])

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=scatter_data, x='tenure_months', y='monthly_spend', hue='churn_label',
                palette=CHURN_COLOURS, alpha=0.6, s=35, edgecolor='white', linewidth=0.3, ax=ax)
ax.set_title('Tenure versus monthly spend, by churn status')
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Monthly spend (£)')
ax.legend(title='Status', frameon=False)

In [ ]:
numeric_cols = ['tenure_months', 'monthly_spend', 'support_tickets', 'churned']
pearson = df[numeric_cols].corr(method='pearson')
spearman = df[numeric_cols].corr(method='spearman')

fig, (ax_p, ax_s) = plt.subplots(1, 2, figsize=(11, 4.5))
sns.heatmap(pearson, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, square=True,
            cbar_kws={'label': 'Correlation'}, ax=ax_p)
ax_p.set_title('Pearson (linear)')
sns.heatmap(spearman, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, square=True,
            cbar_kws={'label': 'Correlation'}, ax=ax_s)
ax_s.set_title('Spearman (rank-based)')
fig.tight_layout()

**Questions to discuss:**

- Where do Pearson and Spearman values disagree? What in the data might cause this?
- Which pair of variables has the strongest association with churn, and is that association the same size under both measures?
- The scatter plot uses `alpha` to handle overlap. Where does overlap make the pattern hard to judge?

### A misleading aggregate

Aggregated associations can mislead. The next cell first looks at churn across all customers by monthly spend quartile, then repeats the comparison within each plan type, using quartiles cut inside each plan. Plan type affects both spend and churn, so the two views may disagree.

In [ ]:
complete = df.dropna(subset=['monthly_spend']).copy()
complete['spend_quartile'] = pd.qcut(complete['monthly_spend'], 4,
                                     labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)'])
# Quartiles computed separately inside each plan type
complete['within_plan_quartile'] = (complete.groupby('plan_type')['monthly_spend']
                                    .transform(lambda s: pd.qcut(s, 4, labels=False)))

overall = complete.groupby('spend_quartile', observed=True)['churned'].mean()
by_plan = (complete.groupby(['within_plan_quartile', 'plan_type'])['churned']
           .mean().unstack('plan_type').reindex(columns=PLAN_ORDER))
counts = pd.crosstab(complete['within_plan_quartile'], complete['plan_type']).reindex(columns=PLAN_ORDER)
print('Customers per within-plan quartile (columns = plan type):')
print(counts.to_string())

fig, (ax_all, ax_plan) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

ax_all.bar(overall.index.astype(str), overall.values, color=OKABE_ITO[0])
ax_all.set_title('All customers, overall quartiles')
ax_all.set_xlabel('Monthly spend quartile (all customers)')
ax_all.set_ylabel('Churn rate (proportion)')

quartile_names = [f'Q{i + 1}' for i in range(4)]
for plan in PLAN_ORDER:
    ax_plan.plot(quartile_names, by_plan[plan], marker='o', linewidth=2,
                 color=PLAN_COLOURS[plan], label=plan)
ax_plan.set_title('Within each plan type, quartiles inside the plan')
ax_plan.set_xlabel('Monthly spend quartile (within plan type)')
ax_plan.legend(title='Plan', frameon=False)
fig.tight_layout()

**Questions to discuss:**

- Describe the pattern across all customers. Describe it again within each plan type. Does it change?
- Some points in the within-plan view rest on few customers. How would you show that uncertainty?
- What third variable could explain any difference between the two views? What would you need to check before saying spend affects churn?
- What is the difference between an association you observe and a causal effect you can claim?

## 5. Multivariate exploration

To explore more than two variables, use:

- **Hue** (colour) to separate groups, such as churn status;
- **Faceting** (small multiples) to split a chart by a category, such as plan type, with the same axes in every panel;
- **Pairplots** on a small set of numeric columns, to scan many pairwise relationships at once.

Keep the number of encodings small. Four or five simultaneous channels overwhelm the reader. Avoid **3D charts**: perspective distorts lengths, points hide one another, and values are hard to read. Small multiples with shared axes communicate the same information more clearly.

A pairplot on four variables, using a random sample of 200 complete records:

In [ ]:
pair_cols = ['tenure_months', 'monthly_spend', 'support_tickets', 'churn_label']
pair_sample = df[pair_cols].dropna().sample(n=200, random_state=42)

grid = sns.pairplot(pair_sample, hue='churn_label', palette=CHURN_COLOURS, corner=True, height=1.8,
                    plot_kws={'alpha': 0.6, 's': 18, 'edgecolor': 'none'})
grid.figure.suptitle('Pairwise view of four variables (200-customer sample)', y=1.02)

Next, a faceted view. Each panel is one plan type, with the same y-axis, showing the churn rate by contract type. The error bars show 95% confidence intervals, so you can see how precise each estimate is.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)

for ax, plan in zip(axes, PLAN_ORDER):
    subset = df[(df['plan_type'] == plan)].dropna(subset=['contract_type'])
    sns.barplot(data=subset, x='contract_type', y='churned', order=CONTRACT_ORDER,
                color=PLAN_COLOURS[plan], errorbar=('ci', 95), ax=ax)
    ax.set_title(f'{plan} plan')
    ax.set_xlabel('Contract type')
    ax.tick_params(axis='x', rotation=20)

axes[0].set_ylabel('Churn rate (proportion)')
fig.tight_layout()

**Questions to discuss:**

- Does the contract-type pattern look the same in every plan type? Where would you be cautious about drawing conclusions?
- Which of the pairwise panels in the pairplot show a difference between churned and retained customers? Are those differences large relative to the spread within each group?
- What would a 3D scatter of these four variables hide that the pairplot shows?

## 6. Interactive exploration with Plotly

Static charts fix every decision in advance. Interactive charts let you investigate:

- **Hover** reveals the record behind a point, so you can identify unusual customers.
- **Zoom and pan** let you examine dense regions and extreme values.
- **Legend clicks** hide or isolate groups, so you can compare them directly.

Interactive charts suit exploration. For a report, a static figure with a clear caption is usually easier to check and reproduce.

The first chart plots tenure against monthly spend. Hover over the points that look unusual and note their details.

In [ ]:
plot_df = df.dropna(subset=['monthly_spend', 'tenure_months', 'support_tickets'])

fig = px.scatter(
    plot_df, x='tenure_months', y='monthly_spend', color='churn_label',
    color_discrete_map=CHURN_COLOURS, opacity=0.7,
    hover_data=['customer_id', 'plan_type', 'contract_type', 'support_tickets'],
    labels={'tenure_months': 'Tenure (months)', 'monthly_spend': 'Monthly spend (£)',
            'churn_label': 'Status', 'support_tickets': 'Support tickets'},
    title='Tenure versus monthly spend (hover for details, click legend to filter)',
    template='plotly_white',
)
fig.show()

In [ ]:
fig_box = px.box(
    plot_df, x='plan_type', y='monthly_spend', color='churn_label',
    category_orders={'plan_type': PLAN_ORDER}, color_discrete_map=CHURN_COLOURS,
    points='outliers',
    labels={'plan_type': 'Plan type', 'monthly_spend': 'Monthly spend (£)', 'churn_label': 'Status'},
    title='Monthly spend by plan type and churn status',
    template='plotly_white',
)
fig_box.show()

**Try these interactions and note what each one reveals:**

- Hover over the most extreme spend values. Which plan types and contracts do they belong to? Does this look like genuine spend or a data error?
- Click a legend entry to hide one churn group. Does the remaining group look different once the other is removed?
- Zoom into a dense region of the scatter plot. Can you see structure that was hidden at full scale?
- In the box plot, hover over a box to read its quartiles. Which values are exact, and which are read by eye from a static chart?

## 7. Responsible visual communication

A chart is an argument. Before you share one, check that it:

- uses **accessible colour**: the Okabe–Ito palette, sequential palettes for ordered values, and never relies on colour alone to carry meaning;
- has **labels with units**, a clear title that states the message, and a sample size;
- does **not use truncated axes** for bars, and states the scale for scatter plots;
- shows **uncertainty** rather than a bare point estimate, where the group sizes are small;
- states the **limitations** of the data, here including that it is synthetic.

The next chart shows the churn rate for each contract type with 95% confidence intervals, using a normal approximation for a proportion. The interval is wider when a group is small.

In [ ]:
rates = (df.dropna(subset=['contract_type'])
           .groupby('contract_type')['churned']
           .agg(rate='mean', n='count')
           .reindex(CONTRACT_ORDER))
rates['margin'] = 1.96 * np.sqrt(rates['rate'] * (1 - rates['rate']) / rates['n'])

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(rates.index, rates['rate'], yerr=rates['margin'], fmt='o', markersize=8,
            color=OKABE_ITO[0], ecolor=OKABE_ITO[0], capsize=6, linewidth=2)
for x, (rate, n_group) in enumerate(zip(rates['rate'], rates['n'])):
    ax.annotate(f'n = {n_group}', (x, rate), textcoords='offset points', xytext=(0, 14), ha='center', fontsize=9)
ax.set_ylim(0, min(1, (rates['rate'] + rates['margin']).max() * 1.2))
ax.set_title('Churn rate by contract type, with 95% confidence intervals')
ax.set_xlabel('Contract type')
ax.set_ylabel('Churn rate (proportion)')

**Questions to discuss:**

- If two intervals overlap, what does that imply about the difference between the groups? What does it not imply?
- Which annotation would you add for a reader who has never seen a confidence interval before?
- What must be stated about this dataset before any of these charts is shown to a business audience?

## 8. Guided mini-project

Work in pairs or alone. Investigate churn by plan type, tenure, monthly spend and support tickets. Each question has starter code with `TODO` markers. The cells run as they are, but the placeholder titles must be replaced and the analysis completed.

For each question, make a decision explicit: how you handle missing values, what you compare, and which chart you use. A good answer makes one supported claim and states what would change your mind.

In [ ]:
# Question 1 (group comparison): Does the churn rate differ by plan type?
# TODO 1: decide how to handle missing values for this question, and explain your choice in a comment.
# TODO 2: compute the churn rate per plan type, with a measure of uncertainty and group sizes.
# TODO 3: draw a chart that makes the comparison clear; label axes with units or proportions.
fig, ax = plt.subplots(figsize=(7, 4))
ax.set_title('TODO: a title that states the message')
ax.set_xlabel('TODO: plan type')
ax.set_ylabel('TODO: churn measure, with units')

In [ ]:
# Question 2 (relationship): Is the number of support tickets associated with churn,
# and does that association persist within contract types?
# TODO 1: choose a relationship or group-level chart that suits a binary outcome.
# TODO 2: check the association within each contract type, not only overall.
# TODO 3: label the axes and state how many customers each point or bar represents.
fig, ax = plt.subplots(figsize=(8, 4))
ax.set_title('TODO: a title that states the message')

In [ ]:
# Question 3 (interactive view): Which customers stand out on tenure and monthly spend,
# and what do their details reveal?
# TODO 1: build a Plotly chart with hover_data that includes customer_id and plan_type.
# TODO 2: use colour for churn_label, and set a clear title and axis labels.
# TODO 3: after showing the chart, write down which records you investigated and why.
fig = None

## 9. Findings and reflection

### Findings template

Complete this template in your own words. Every finding must cite evidence from a chart or table in this notebook, and must separate what you observed from any causal claim.

**Finding 1:** [claim]
- Evidence: [chart or table, with the relevant numbers]
- Uncertainty or caveat: [group sizes, intervals, outliers or missing data]

**Finding 2:** [claim]
- Evidence: [chart or table, with the relevant numbers]
- Uncertainty or caveat: [ ]

**Finding 3:** [claim]
- Evidence: [chart or table, with the relevant numbers]
- Uncertainty or caveat: [ ]

**Limitation 1:** [ ]

**Limitation 2:** [ ]

**Recommended next analytical step 1:** [ ]

**Recommended next analytical step 2:** [ ]

### Reflection questions

1. What is the difference between exploring a dataset and presenting one supported claim? Which parts of this session were which?
2. Which data-quality decision did you find hardest, and what evidence would have changed your mind?
3. Where did an aggregated pattern differ from a within-group pattern, and how would you report that to a non-specialist?
4. Which of your charts would be misleading without its uncertainty or sample size?
5. What would you need to know about real customers before any of these findings could inform a business decision?